# Show Reel — Community Persona Pipeline (Vertex AI **Batch** + **Multimodal**)

**Platform:** Instagram   **Runtime:** Google Colab Enterprise / Vertex AI (ADC auth)

This is the batch-inference rewrite of the persona pipeline. Both LLM stages now run as
**asynchronous Vertex AI batch jobs** (unified `google-genai` SDK): build a JSONL of requests →
upload to GCS → `client.batches.create` → poll → retrieve the sharded JSONL responses. Batch is
~50% cheaper than online calls and sidesteps per-request rate limits.

Persona identification is **multimodal**: each user request carries the media of the posts they
engage with most, read straight from GCS via `gs://` `fileData` parts —
- **image / carousel** posts → the photo(s) + metadata (tagged users, music, caption);
- **reel / feed / carousel_video** posts → the sampled `frames/*.jpg` + the `transcription.txt` + metadata.

**Flow:** Stage 0 features → Stage 1 taxonomy discovery (Flash, batch) → human approval →
Stage 2 classification (Pro, batch). Set `PIPELINE_MODE` in config.

## Install Dependencies

Run once per runtime, then restart the kernel. The unified `google-genai` SDK drives both the
batch jobs and the online connectivity test; `google-cloud-storage` handles GCS upload/list/download.

In [4]:
!pip install -q google-genai google-cloud-storage gcsfs pandas pyarrow tqdm emoji

## Configuration

Everything environment- and cost-related lives here. The multimodal caps (`MAX_MEDIA_POSTS_PER_USER`,
`MAX_IMAGES_PER_POST`) directly drive token cost — raise them for richer context, lower them to save money.

In [ ]:
import os

# ============================ PLATFORM TOGGLE ============================
# Run the whole persona pipeline against a different platform by changing this.
# Comments come from the multi-platform prepared exports and are filtered to PLATFORM.
PLATFORM = "instagram"        # instagram | youtube | tiktok | facebook
# ========================================================================

# GCP / Vertex AI
GCP_PROJECT_ID = "gen-lang-client-0792749758"
GCP_LOCATION   = "us-central1"
GCS_BUCKET     = "afb_showreel"

# --- Prepared comments (ALL platforms; filtered to PLATFORM at load) -----------
# comments_ml      : numeric features + bare-numeric author_id/media_id + comment_id (ig_comment_<n>)
# edges_replies_to : reply structure (src --replies to--> dst), joined on comment_id
# comments_llm     : raw text keyed by comment_id (no author_id -> join via comment_id)
COMMENTS_ML_PATH   = f"gs://{GCS_BUCKET}/Preped_Comments/comments_ml.parquet"
COMMENTS_LLM_PATH  = f"gs://{GCS_BUCKET}/Preped_Comments/comments_llm.jsonl"
EDGES_REPLIES_PATH = f"gs://{GCS_BUCKET}/HeteroGraph/edges_replies_to.parquet"

# --- Multimodal context (Instagram only): post metadata + media on GCS ---------
# ig_multimodal_final bridges media_id <-> shortcode and carries post metadata
# (tagged users, music, caption); media objects live under MEDIA_GCS_PREFIX.
MEDIA_INDEX_PATH = f"gs://{GCS_BUCKET}/ig_multimodal_final.parquet"
MEDIA_GCS_PREFIX = "multimodal_dataset_fixed/"     # gs://bucket/<prefix>/<form>/<shortcode>/...
ATTACH_MEDIA     = (PLATFORM == "instagram")        # media context is wired for IG; others run comments-only

# Models — Flash discovers the taxonomy on a sample, Pro classifies the full set.
MODEL_STAGE1_EXPLORATORY = "gemini-2.5-flash"
MODEL_STAGE2_CLASSIFY    = "gemini-2.5-pro"

# Determinism (academic reproducibility)
TEMPERATURE              = 0.0
TOP_P                    = 1.0
# Gemini 2.5 are THINKING models: thinking tokens count against maxOutputTokens, so these must be
# big enough for (thinking + JSON output) or the response truncates (finishReason=MAX_TOKENS).
MAX_OUTPUT_TOKENS_STAGE1 = 8192
MAX_OUTPUT_TOKENS_STAGE2 = 2048
THINKING_BUDGET_STAGE1   = 1024   # Flash: cap thinking so the JSON array always completes (0 = disable)
THINKING_BUDGET_STAGE2   = 512    # Pro:   min 128; keep modest for cost across many requests

# Pipeline mode:  SAMPLE = Stage 0+1 (discover, then pause for approval)
#                 FULL   = Stage 0+2 (classify, needs APPROVED taxonomy)
#                 ALL    = both in sequence
PIPELINE_MODE = "SAMPLE"

# Sampling / batching
SAMPLE_N_USERS           = 10000
SAMPLE_SEED              = 42
STAGE1_USERS_PER_REQUEST = 3
STAGE2_MAX_USERS         = None

# Multimodal attachment caps (cost control)
MAX_MEDIA_POSTS_PER_USER = 2
MAX_IMAGES_PER_POST      = 2
INCLUDE_TRANSCRIPT       = True
MAX_TRANSCRIPT_CHARS     = 1500

# Batch I/O on GCS
BATCH_INPUT_PREFIX    = "persona_batch/input/"
BATCH_OUTPUT_PREFIX   = "persona_batch/output/"
POLL_INTERVAL_SECONDS = 60

# Local artifacts (uploaded to GCS after the run)
LOCAL_DIR          = "outputs"
TAXONOMY_JSON_PATH = f"{LOCAL_DIR}/taxonomy.json"
RESULTS_PATH       = f"{LOCAL_DIR}/user_personas.parquet"
os.makedirs(LOCAL_DIR, exist_ok=True)

print("✅ Configuration loaded.")
print(f"   Platform: {PLATFORM}  (media context: {'ON' if ATTACH_MEDIA else 'OFF'})")
print(f"   Bucket:   gs://{GCS_BUCKET}")
print(f"   Mode:     {PIPELINE_MODE}")

✅ Configuration loaded.
   Platform: instagram  (media context: ON)
   Bucket:   gs://afb_showreel
   Mode:     SAMPLE


## Vertex AI Client (google-genai)

`genai.Client(vertexai=True, ...)` flips the unified SDK to the Vertex backend (IAM/ADC auth, GCS
I/O, regional endpoints) — the same client submits batch jobs and runs the online connectivity test.

In [2]:
from google import genai
from google.genai.types import CreateBatchJobConfig, JobState

client = genai.Client(vertexai=True, project=GCP_PROJECT_ID, location=GCP_LOCATION)

def gen_config_dict(max_tokens: int, thinking_budget: int | None = None) -> dict:
    """Per-request generationConfig (camelCase REST keys) embedded in each JSONL line.
    responseMimeType forces JSON; thinkingConfig caps the 2.5 models' thinking-token spend so
    the visible output isn't starved (the MAX_TOKENS truncation bug)."""
    cfg = {
        "temperature": TEMPERATURE,
        "topP": TOP_P,
        "maxOutputTokens": max_tokens,
        "responseMimeType": "application/json",
    }
    if thinking_budget is not None:
        cfg["thinkingConfig"] = {"thinkingBudget": thinking_budget}
    return cfg

print("✅ genai Vertex client ready:", GCP_PROJECT_ID, GCP_LOCATION)

✅ genai Vertex client ready: gen-lang-client-0792749758 us-central1


### Optional — Model Connectivity Test

Confirms both models are reachable (online call) before committing to a long batch job.

In [3]:
for stage, model_name in [("Stage 1", MODEL_STAGE1_EXPLORATORY), ("Stage 2", MODEL_STAGE2_CLASSIFY)]:
    try:
        r = client.models.generate_content(model=model_name, contents="Reply with the single word: OK")
        print(f"✅ {stage} ({model_name}): {r.text.strip()[:40]}")
    except Exception as e:
        print(f"❌ {stage} ({model_name}): {e}")

✅ Stage 1 (gemini-2.5-flash): OK
✅ Stage 2 (gemini-2.5-pro): OK


## Data Loading

Persona features come from the **comments dataset** — the multi-platform prepared exports, filtered
to `PLATFORM`:
- `Preped_Comments/comments_ml.parquet` — per-comment numeric features + bare-numeric `author_id`/`media_id`
- `HeteroGraph/edges_replies_to.parquet` — reply structure (`src --replies to--> dst`), joined on `comment_id`

For Instagram, `ig_multimodal_final.parquet` (post metadata + `media_id`↔`shortcode`) and the GCS media
are loaded as **multimodal context** only. Other platforms run comments-only.

In [4]:
import pandas as pd

print(f"Loading prepared comments for platform = {PLATFORM!r}")

# 1) Comment feature matrix — predicate-pushdown filter to PLATFORM (skips other platforms).
ig_comments = pd.read_parquet(COMMENTS_ML_PATH, filters=[("platform", "==", PLATFORM)])
ig_comments["author_id"] = ig_comments["author_id"].astype(str)
ig_comments["media_id"]  = ig_comments["media_id"].astype(str)
print(f"  comments_ml[{PLATFORM}]: {len(ig_comments):,} comments | "
      f"{ig_comments['author_id'].nunique():,} authors | {ig_comments['media_id'].nunique():,} media")

# 2) Reply structure: edges_replies_to (src --replies to--> dst), joined on comment_id.
try:
    replies = pd.read_parquet(EDGES_REPLIES_PATH, filters=[("platform", "==", PLATFORM)],
                              columns=["src_comment_id", "dst_comment_id"])
    replies = (replies.rename(columns={"src_comment_id": "comment_id",
                                       "dst_comment_id": "reply_to_comment_id"})
                      .drop_duplicates("comment_id"))
    ig_comments = ig_comments.merge(replies, on="comment_id", how="left")
    print(f"  reply edges: {len(replies):,} | replies flagged on "
          f"{ig_comments['reply_to_comment_id'].notna().sum():,} comments")
except Exception as e:
    print("  ⚠️  edges_replies_to unavailable -> no reply features:", str(e)[:120])
    ig_comments["reply_to_comment_id"] = pd.NA

# 3) Instagram-only multimodal context: post metadata + media index.
if ATTACH_MEDIA:
    media_index = pd.read_parquet(MEDIA_INDEX_PATH)
    media_index = media_index[media_index["media_id"].notna()].copy()
    media_index["media_id"] = media_index["media_id"].astype(str)
    ig_media = media_index                      # post-metadata source for Stage 0 temporal features
    print(f"  media_index: {len(media_index):,} posts | {media_index['shortcode'].nunique():,} shortcodes")
else:
    media_index = None
    ig_media = None
    print(f"  media context OFF for platform={PLATFORM} -> comments-only run")

# 4) Validation
n_before = len(ig_comments)
ig_comments = ig_comments[ig_comments["author_id"].notna() & (ig_comments["author_id"] != "")].copy()
print(f"\n✅ Comments ready: {len(ig_comments):,}/{n_before:,} | "
      f"unique authors: {ig_comments['author_id'].nunique():,}")

Loading prepared comments for platform = 'instagram'
  comments_ml[instagram]: 499,752 comments | 194,513 authors | 1,501 media
  reply edges: 38,092 | replies flagged on 38,092 comments
  media_index: 1,493 posts | 1,493 shortcodes

✅ Comments ready: 499,752/499,752 | unique authors: 194,513


## Stage 0 — Feature Engineering

All comment-level features (word count, emoji count, etc.) are pre-computed upstream. Here we derive
binary flags + temporal features, aggregate to one row per `author_id`, and attach each user's
representative comment text.

In [5]:
import numpy as np
from datetime import timedelta

# 1. Binary flags from pre-computed counts
ig_comments["has_emoji"]    = (ig_comments["emoji_count"] > 0).astype(int)
ig_comments["has_question"] = (ig_comments["question_count"] > 0).astype(int)
ig_comments["has_exclaim"]  = (ig_comments["exclamation_count"] > 0).astype(int)
ig_comments["is_reply"]     = ig_comments["reply_to_comment_id"].notna().astype(int)
print("✅ Binary flags derived.")

# 2. Temporal features vs. post time (uses Instagram post metadata when available)
ig_comments["timestamp"] = pd.to_datetime(ig_comments["timestamp"], errors="coerce", utc=True)
if ATTACH_MEDIA and ig_media is not None and "timestamp" in ig_media.columns:
    pm = ig_media[["media_id", "timestamp"]].rename(columns={"timestamp": "post_timestamp"})
    ig_comments = ig_comments.merge(pm, on="media_id", how="left")
    ig_comments["post_timestamp"]  = pd.to_datetime(ig_comments["post_timestamp"], errors="coerce", utc=True)
    ig_comments["hours_to_comment"] = (
        (ig_comments["timestamp"] - ig_comments["post_timestamp"]).dt.total_seconds() / 3600
    ).clip(lower=0)
else:
    ig_comments["hours_to_comment"] = np.nan
    print("   (no post metadata -> hours_to_comment = NaN)")

# 3. User-level aggregation
def build_user_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    grp = df.groupby("author_id")
    agg = {
        "total_comments":          grp.size(),
        "unique_posts_commented":  grp["media_id"].nunique(),
        "total_replies_made":      grp["is_reply"].sum(),
        "reply_ratio":             grp["is_reply"].mean(),
        "mean_hours_to_comment":   grp["hours_to_comment"].mean(),
        "median_hours_to_comment": grp["hours_to_comment"].median(),
        "pct_comments_under_1h":   grp["hours_to_comment"].apply(lambda x: (x < 1).mean()),
        "pct_comments_under_24h":  grp["hours_to_comment"].apply(lambda x: (x < 24).mean()),
        "activity_span_days":      grp["timestamp"].apply(lambda x: (x.max() - x.min()).days if x.notna().any() else 0),
        "mean_word_count":         grp["word_count"].mean(),
        "mean_mention_count":      grp["mention_count"].mean(),
        "emoji_usage_rate":        grp["has_emoji"].mean(),
        "question_rate":           grp["has_question"].mean(),
        "exclamation_rate":        grp["has_exclaim"].mean(),
    }
    feat = pd.DataFrame(agg).reset_index()
    feat["post_concentration_ratio"] = (
        feat["unique_posts_commented"] / feat["total_comments"]
    ).clip(upper=1.0)
    # Fill timing features that are NaN when post metadata is absent (non-IG platforms).
    for c in ["mean_hours_to_comment", "median_hours_to_comment",
              "pct_comments_under_1h", "pct_comments_under_24h"]:
        feat[c] = feat[c].fillna(0)
    return feat

user_features = build_user_feature_matrix(ig_comments)
print(f"✅ User feature matrix built: {len(user_features):,} users.")

# 4. Attach representative comment text from comments_llm (filter to PLATFORM, join by comment_id)
print(f"Loading comment text from {COMMENTS_LLM_PATH} ...")
_text_chunks = []
for _chunk in pd.read_json(COMMENTS_LLM_PATH, lines=True, chunksize=200_000):
    if "platform" in _chunk.columns:
        _chunk = _chunk[_chunk["platform"] == PLATFORM]
    if len(_chunk):
        _text_chunks.append(_chunk[["comment_id", "text"]])
llm_text = (pd.concat(_text_chunks, ignore_index=True)
            if _text_chunks else pd.DataFrame(columns=["comment_id", "text"]))

txt = ig_comments[["comment_id", "author_id"]].merge(llm_text, on="comment_id", how="inner")
top_comments = (
    txt.groupby("author_id").head(5)
       .groupby("author_id")["text"]
       .apply(lambda ts: " ||| ".join(ts.astype(str).tolist()))
       .reset_index()
       .rename(columns={"text": "top_comments_sample"})
)
user_features = user_features.merge(top_comments, on="author_id", how="left")
print(f"✅ Representative comment history attached for {top_comments.shape[0]:,} users.")

✅ Binary flags derived.
✅ User feature matrix built: 194,513 users.
Loading comment text from gs://afb_showreel/Preped_Comments/comments_llm.jsonl ...
✅ Representative comment history attached for 194,513 users.


## Feature Selection for Clustering

Below are **all available features** from the comment-level data and user-level aggregations.

In [ ]:
# ALL AVAILABLE FEATURES FROM STAGE 0
all_available = {
    "Volume & Breadth": [
        "total_comments",
        "unique_posts_commented",
        "activity_span_days",
    ],
    "Reply Behavior": [
        "total_replies_made",
        "reply_ratio",
    ],
    "Timing & Recency": [
        "mean_hours_to_comment",
        "median_hours_to_comment",
        "pct_comments_under_1h",
        "pct_comments_under_24h",
    ],
    "Textual Style": [
        "mean_word_count",
        "mean_mention_count",
        "emoji_usage_rate",
        "question_rate",
        "exclamation_rate",
    ],
    "Concentration": [
        "post_concentration_ratio",
    ],
}

# Display grouped
total_count = 0
for category, features in all_available.items():
    print(f"
{category}:")
    for f in features:
        avail = f'✓' if f in user_features.columns else '✗ (missing)'
        print(f"  {avail}  {f}")
        total_count += 1

print(f"
Total available: {total_count}")

In [ ]:
# EDIT THIS LIST: Remove or comment out features you don't want in clustering
SELECTED_NUMERIC_FEATURES = [
    "total_comments",
    "unique_posts_commented",
    "activity_span_days",
    # "total_replies_made",  # <- REDUNDANT: use reply_ratio instead
    "reply_ratio",
    "mean_hours_to_comment",
    # "median_hours_to_comment",  # <- REDUNDANT: use mean instead
    "pct_comments_under_1h",
    # "pct_comments_under_24h",  # <- OPTIONAL: tight subset of 1h signal
    "mean_word_count",
    "mean_mention_count",
    "emoji_usage_rate",
    "question_rate",
    "exclamation_rate",
    "post_concentration_ratio",
]

# Filter to only columns that exist
SELECTED_NUMERIC_FEATURES = [c for c in SELECTED_NUMERIC_FEATURES if c in user_features.columns]

print(f"
✅ Selected {len(SELECTED_NUMERIC_FEATURES)} numeric features:")
for f in SELECTED_NUMERIC_FEATURES:
    print(f"   • {f}")

## Multimodal Media Context

Builds the bridge from a commenter to the **media they engage with**:

1. `user_top_posts` — each `author_id`'s top-N most-commented `media_id`s.
2. media index maps — `media_id → shortcode` and `shortcode → post metadata`.
3. `build_media_manifest()` lists the GCS media tree **once** and groups every `gs://` object by
   shortcode into `images` (jpg/png) and `transcripts` (transcription.txt).
4. `build_user_media_parts(author_id)` emits the multipart payload (metadata text + image `fileData`
   parts + transcript text) that gets attached to that user's Stage 1 / Stage 2 request.

In [6]:
import numpy as np
from google.cloud import storage

if not ATTACH_MEDIA:
    # Non-Instagram platform: no media context — requests are text-only.
    user_top_posts = {}
    media_images, media_transcripts = {}, {}
    def build_user_media_parts(author_id) -> list:
        return []
    print(f"⏭  Media context disabled for platform={PLATFORM}; requests will be text-only.")
else:
    def _norm_id(x):
        """media_id can arrive as int, float (123.0) or str — normalise to a bare digit string."""
        s = str(x)
        return s[:-2] if s.endswith(".0") else s

    # 1) media index maps (bare-numeric media_id matches comments_ml.media_id)
    _mi = media_index[media_index["media_id"].notna()].copy()
    _mi["media_id"] = _mi["media_id"].map(_norm_id)
    _mediaid_to_shortcode = dict(zip(_mi["media_id"], _mi["shortcode"]))
    _meta_by_shortcode    = {r["shortcode"]: r for r in _mi.to_dict("records")}

    # 2) per-user top engaged posts (by comment volume)
    _tp = ig_comments[["author_id", "media_id"]].dropna().copy()
    _tp["author_id"] = _tp["author_id"].astype(str)
    _tp["media_id"]  = _tp["media_id"].map(_norm_id)
    _counts = (
        _tp.groupby(["author_id", "media_id"]).size()
           .reset_index(name="n")
           .sort_values(["author_id", "n"], ascending=[True, False])
    )
    user_top_posts = (
        _counts.groupby("author_id")["media_id"]
               .apply(lambda s: list(s.head(MAX_MEDIA_POSTS_PER_USER)))
               .to_dict()
    )
    print(f"✅ user_top_posts computed for {len(user_top_posts):,} users.")

    # 3) GCS media manifest — one list pass over the whole media tree
    def build_media_manifest(bucket=GCS_BUCKET, prefix=MEDIA_GCS_PREFIX):
        c = storage.Client(project=GCP_PROJECT_ID)
        images, transcripts = {}, {}
        for blob in c.list_blobs(bucket, prefix=prefix):
            name = blob.name
            low  = name.lower()
            segs = name.split("/")          # multimodal_dataset_fixed/<form>/<shortcode>/...
            if len(segs) < 3:
                continue
            sc  = segs[2]
            uri = f"gs://{bucket}/{name}"
            if low.endswith((".jpg", ".jpeg", ".png")):
                images.setdefault(sc, []).append(uri)
            elif low.endswith(".txt") and "transcri" in low:
                transcripts.setdefault(sc, []).append(uri)
        for d in (images, transcripts):
            for k in d:
                d[k].sort()
        print(f"✅ manifest: {len(images):,} posts with images, {len(transcripts):,} with transcripts.")
        return images, transcripts

    media_images, media_transcripts = build_media_manifest()

    # 4) builders
    _transcript_cache = {}
    def fetch_transcript(sc: str) -> str:
        if sc in _transcript_cache:
            return _transcript_cache[sc]
        chunks = []
        c = storage.Client(project=GCP_PROJECT_ID)
        for uri in media_transcripts.get(sc, [])[:8]:     # carousel_video has one per slide
            try:
                chunks.append(storage.Blob.from_string(uri, client=c).download_as_text())
            except Exception:
                pass
        txt = " ".join(x.strip() for x in chunks if x.strip())
        _transcript_cache[sc] = txt
        return txt

    def _pick_images(sc: str, k: int):
        """Evenly sample k image URIs across the post (spread frames/slides, not the first k)."""
        uris = media_images.get(sc, [])
        if len(uris) <= k:
            return uris
        step = len(uris) / k
        return [uris[int(i * step)] for i in range(k)]

    def _txt(v):
        return v.strip() if isinstance(v, str) else ""

    def _as_list_str(v):
        if isinstance(v, (list, tuple, np.ndarray)):
            return ", ".join(str(x) for x in v if str(x) not in ("nan", "None", ""))
        return _txt(v)

    def format_post_meta_text(sc: str) -> str:
        row  = _meta_by_shortcode.get(sc, {})
        cf   = _txt(row.get("content_form"))
        bits = [f"POST [{cf or 'post'}] shortcode={sc}"]
        cap  = _txt(row.get("caption"))
        if cap:
            bits.append(f"caption: {cap[:300]}")
        tg = _as_list_str(row.get("tagged_usernames"))
        if tg:
            bits.append(f"tagged: {tg}")
        co = _as_list_str(row.get("coauthors"))
        if co:
            bits.append(f"coauthors: {co}")
        song, artist, atype = _txt(row.get("song_title")), _txt(row.get("artist")), _txt(row.get("audio_type"))
        if song or artist:
            bits.append(f"music: {song} - {artist} ({atype})")
        loc = _txt(row.get("location_name"))
        if loc:
            bits.append(f"location: {loc}")
        return " | ".join(bits)

    def build_user_media_parts(author_id) -> list:
        """Multipart payload for a user's most-engaged posts:
        image/carousel -> photo fileData + metadata; reel/feed/carousel_video -> frame fileData + transcript + metadata."""
        parts = []
        for media_id in user_top_posts.get(str(author_id), []):
            sc = _mediaid_to_shortcode.get(_norm_id(media_id))
            if not sc or sc not in media_images:
                continue
            parts.append({"text": format_post_meta_text(sc)})
            for uri in _pick_images(sc, MAX_IMAGES_PER_POST):
                parts.append({"fileData": {"fileUri": uri, "mimeType": "image/jpeg"}})
            if INCLUDE_TRANSCRIPT:
                t = fetch_transcript(sc)
                if t:
                    parts.append({"text": "transcript: " + t[:MAX_TRANSCRIPT_CHARS]})
        return parts

    _demo = next((a for a in user_features["author_id"].astype(str) if build_user_media_parts(a)), None)
    print("✅ builders ready. Example user media parts:",
          len(build_user_media_parts(_demo)) if _demo else 0)

d:\conda_envs\ma_env\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ user_top_posts computed for 194,513 users.
✅ manifest: 1,473 posts with images, 367 with transcripts.
✅ builders ready. Example user media parts: 8


## Batch Infrastructure

The five reusable stages of a Vertex batch job, lifted from `Vertex_Batch_Inference.ipynb`:
**write JSONL → upload → submit → poll → retrieve+parse**. Submit is non-blocking; we re-GET the job
each poll until a terminal state; outputs may be sharded across files so we iterate.

In [7]:
import json, os, time, datetime, re
from google.cloud import storage

def strip_fences(s: str) -> str:
    s = s.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    return s.strip()

def write_jsonl(lines: list, path: str) -> str:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for obj in lines:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")
    print(f"[prep] wrote {len(lines):,} requests -> {path}")
    return path

def upload_to_gcs(local_path: str, bucket_name: str, blob_name: str) -> str:
    c = storage.Client(project=GCP_PROJECT_ID)
    c.bucket(bucket_name).blob(blob_name).upload_from_filename(local_path)
    uri = f"gs://{bucket_name}/{blob_name}"
    print(f"[upload] {local_path} -> {uri}")
    return uri

def submit_batch_job(input_uri: str, output_uri: str, model: str):
    """Non-blocking submit. dest is a PREFIX; Vertex writes a unique run subfolder under it."""
    job = client.batches.create(
        model=model, src=input_uri,
        config=CreateBatchJobConfig(dest=output_uri),
    )
    print(f"[submit] {model} -> {job.name}  ({job.state})")
    return job

def poll_until_complete(job_name: str):
    terminal = {JobState.JOB_STATE_SUCCEEDED, JobState.JOB_STATE_FAILED,
                JobState.JOB_STATE_CANCELLED, JobState.JOB_STATE_PAUSED}
    while True:
        job = client.batches.get(name=job_name)
        print(f"[poll {datetime.datetime.now():%H:%M:%S}] state = {job.state}")
        if job.state in terminal:
            return job
        time.sleep(POLL_INTERVAL_SECONDS)

def retrieve_response_texts(job, bucket_name: str) -> list:
    """Download every output shard and return the model's text per row (None for failed rows)."""
    out_loc = job.dest.gcs_uri
    prefix  = out_loc.replace(f"gs://{bucket_name}/", "")
    c  = storage.Client(project=GCP_PROJECT_ID)
    bk = c.bucket(bucket_name)
    blobs = [b for b in bk.list_blobs(prefix=prefix) if b.name.endswith(".jsonl")]
    texts = []
    for blob in blobs:
        for line in blob.download_as_text().splitlines():
            if not line.strip():
                continue
            rec  = json.loads(line)
            resp = rec.get("response")
            if not resp:
                texts.append(None)               # row-level failure / safety block
                continue
            try:
                parts = resp["candidates"][0]["content"]["parts"]
                texts.append("".join(p.get("text", "") for p in parts))
            except (KeyError, IndexError):
                texts.append(None)
    print(f"[retrieve] {len(texts):,} response rows from {len(blobs)} shard(s)")
    return texts

def record_batch_job(tag: str, job, dest: str) -> dict:
    """Persist a submitted job so a LATER run (even a fresh kernel) can retrieve it."""
    rec = {"name": job.name, "dest": dest, "state": str(job.state),
           "submitted_at": datetime.datetime.now().isoformat()}
    with open(f"{LOCAL_DIR}/{tag}_job.json", "w", encoding="utf-8") as f:
        json.dump(rec, f, indent=2)
    print(f"[record] {tag} job -> {LOCAL_DIR}/{tag}_job.json")
    return rec

def get_recorded_job(tag: str):
    """Re-fetch the live job for a recorded submission (raises if none recorded)."""
    p = f"{LOCAL_DIR}/{tag}_job.json"
    if not os.path.exists(p):
        raise FileNotFoundError(f"No submitted '{tag}' job recorded at {p} — submit it first.")
    with open(p, encoding="utf-8") as f:
        rec = json.load(f)
    job = client.batches.get(name=rec["name"])
    print(f"[{tag}] {rec['name']} -> {job.state}")
    return job

print("✅ Batch infrastructure ready.")

✅ Batch infrastructure ready.


## Stage 1 — Taxonomy Discovery (Batch · Multimodal)

Groups of `STAGE1_USERS_PER_REQUEST` sampled users go into each JSONL line, every user carrying their
engaged-post media. Flash returns candidate archetypes per line; we concatenate all candidates and
write `taxonomy.json` for **human review** (consolidate → set `status="APPROVED"`).

In [8]:
from tqdm import tqdm

STAGE1_SYSTEM_PROMPT = (
    "You are an expert community analyst for a major Italian influencer agency.\n"
    "Identify distinct audience persona archetypes from Instagram commenter behaviour.\n"
    "Each user profile carries quantitative metrics, a sample of their comments, AND the media\n"
    "(images / video frames + transcript) of the posts they engage with most.\n"
    "Use BOTH how they comment and WHAT content they engage with.\n"
    "For each recurring pattern output a candidate persona with: a short codename (e.g. SUPERFAN),\n"
    "a 1-sentence behavioural description, key quantitative signals, and 2-3 verbatim comment fragments.\n"
    "Output ONLY a valid JSON array. No preamble, no markdown fences.\n"
    'Schema: [{"codename": str, "description": str, "signals": [str], "examples": [str]}]'
)

def format_user_profile_for_stage1(row) -> str:
    return (
        f"USER: {row['author_id']} | "
        f"Total comments: {int(row['total_comments'])} | "
        f"Unique posts: {int(row['unique_posts_commented'])} | "
        f"Activity span: {int(row['activity_span_days'])} days | "
        f"Avg hrs to comment: {row['mean_hours_to_comment']:.1f}h | "
        f"Early commenter (<1h): {row['pct_comments_under_1h']:.0%} | "
        f"Reply ratio: {row['reply_ratio']:.0%} | "
        f"Avg word count: {row['mean_word_count']:.0f} | "
        f"Emoji rate: {row['emoji_usage_rate']:.0%} | "
        f"Question rate: {row['question_rate']:.0%} | "
        f"Sample comments: {str(row.get('top_comments_sample', ''))[:400]}"
    )

def build_stage1_line(group_df) -> dict:
    parts = [{"text": STAGE1_SYSTEM_PROMPT + "\n\n--- USER PROFILES BATCH (with engaged-post media) ---"}]
    for _, row in group_df.iterrows():
        parts.append({"text": "\n" + format_user_profile_for_stage1(row)})
        parts.extend(build_user_media_parts(row["author_id"]))
    parts.append({"text": "\nIdentify all distinct behavioural archetypes in this batch. Output ONLY the JSON array."})
    return {"request": {"contents": [{"role": "user", "parts": parts}],
                        "generationConfig": gen_config_dict(MAX_OUTPUT_TOKENS_STAGE1, THINKING_BUDGET_STAGE1)}}

def save_taxonomy_for_review(candidates, output_path=TAXONOMY_JSON_PATH):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    out = {
        "status": "PENDING_HUMAN_REVIEW",
        "instructions": ("Consolidate the candidates into a MECE taxonomy. Each final persona needs a "
                         "unique 'codename' (plus 'label', 'description', 'quantitative_signals', "
                         "'example_comments'). Set status='APPROVED' before Stage 2."),
        "raw_candidates": candidates,
        "final_taxonomy": [],
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"✅ Raw candidates saved -> {output_path}")
    print("   ⚠️  HUMAN ACTION: review, consolidate, set status='APPROVED'.")

# ── SUBMIT (returns immediately — safe to close the laptop) ────────────────────
def submit_stage1(user_features_df, n_sample=SAMPLE_N_USERS,
                  group_size=STAGE1_USERS_PER_REQUEST, seed=SAMPLE_SEED):
    print(f"\n{'='*60}\nSTAGE 1 (batch submit) — Taxonomy Discovery\n"
          f"  Sample: {n_sample} users | {group_size}/request | Model: {MODEL_STAGE1_EXPLORATORY}\n{'='*60}")
    sample_df = user_features_df.sample(n=min(n_sample, len(user_features_df)),
                                        random_state=seed).reset_index(drop=True)
    groups = [sample_df.iloc[i:i+group_size] for i in range(0, len(sample_df), group_size)]
    lines  = [build_stage1_line(g) for g in tqdm(groups, desc="Build Stage 1 requests")]
    in_uri  = upload_to_gcs(write_jsonl(lines, f"{LOCAL_DIR}/stage1_input.jsonl"),
                            GCS_BUCKET, BATCH_INPUT_PREFIX + "stage1_input.jsonl")
    out_uri = f"gs://{GCS_BUCKET}/{BATCH_OUTPUT_PREFIX}stage1/"
    job = submit_batch_job(in_uri, out_uri, MODEL_STAGE1_EXPLORATORY)
    record_batch_job("stage1", job, out_uri)
    print(f"\n\U0001f4e4 Stage 1 submitted ({len(lines)} requests). Safe to close the laptop.")
    print("   When it finishes, run:  retrieve_stage1()   -> writes taxonomy.json for review")
    return job

# ── RETRIEVE (run later, even from a fresh kernel) ────────────────────────────
def retrieve_stage1(save=True):
    job = get_recorded_job("stage1")
    if job.state != JobState.JOB_STATE_SUCCEEDED:
        print(f"⏳ Stage 1 not ready (state={job.state}). Re-run later.")
        return None
    candidates = []
    for t in retrieve_response_texts(job, GCS_BUCKET):
        if not t:
            continue
        try:
            c = json.loads(strip_fences(t))
            if isinstance(c, list):
                candidates.extend(c)
        except Exception as e:
            print("   parse error:", str(e)[:80])
    print(f"✅ Stage 1 complete. Raw candidates: {len(candidates)}")
    if save:
        save_taxonomy_for_review(candidates)
    return candidates

print("✅ Stage 1 functions ready (submit_stage1 / retrieve_stage1).")

✅ Stage 1 functions ready (submit_stage1 / retrieve_stage1).


### Stage 1 — Inspect Taxonomy Results

In [9]:
if os.path.exists(TAXONOMY_JSON_PATH):
    with open(TAXONOMY_JSON_PATH, "r", encoding="utf-8") as f:
        taxonomy_data = json.load(f)
    status = taxonomy_data.get("status")
    print(f"Taxonomy status: {status}")
    if status == "APPROVED" and taxonomy_data.get("final_taxonomy"):
        display(pd.DataFrame(taxonomy_data["final_taxonomy"]))
    elif taxonomy_data.get("raw_candidates"):
        print("Showing raw candidates (pending review):")
        display(pd.DataFrame(taxonomy_data["raw_candidates"]))
    else:
        print("No candidates found — run Stage 1 first.")
else:
    print("taxonomy.json not found — run Stage 1 first.")

Taxonomy status: PENDING_HUMAN_REVIEW
No candidates found — run Stage 1 first.


## Stage 1.5 — Auto-Consolidate Candidates → Draft Taxonomy

Collapses the raw discovery candidates' naming variants (e.g. `TAGGER` / `THE TAGGER` / `THE_TAGGER`)
into themes, ranks them by how many candidates fall in each, and drafts the top-`k` as a starter
`final_taxonomy` — merged description + signals + example comments per persona.

Still a **draft**: string-normalisation can't merge *semantically* similar themes
(`TAGGER`/`CONNECTOR`, `ADMIRER`/`APPRECIATOR`/`CHEERLEADER`), so review, merge those by hand, trim to
a clean MECE set, drop `_n_candidates`, and set `status="APPROVED"` before Stage 2.

In [17]:
import json, re
from collections import Counter, defaultdict

def draft_taxonomy_from_candidates(path=TAXONOMY_JSON_PATH, top_k=15, min_count=1, write=True):
    """Group raw candidates by naming-normalised codename, rank by frequency, and draft
    the top_k as a starter final_taxonomy (does NOT approve — you review + set status)."""
    data = json.load(open(path, encoding="utf-8"))
    raw = [c for c in data.get("raw_candidates", []) if isinstance(c, dict) and c.get("codename")]
    if not raw:
        print("No raw_candidates — run/retrieve Stage 1 first.")
        return []

    def norm(c):
        c = re.sub(r"[^A-Z0-9 ]", " ", c.upper().replace("_", " "))
        c = re.sub(r"\s+", " ", c).strip()
        return c[4:].strip() if c.startswith("THE ") else c

    groups = defaultdict(list)
    for c in raw:
        groups[norm(c["codename"])].append(c)
    ranked = [(k, v) for k, v in sorted(groups.items(), key=lambda kv: len(kv[1]), reverse=True)
              if len(v) >= min_count][:top_k]

    def merge_list(cands, key, limit=6):
        seen, out = set(), []
        for c in cands:
            for item in (c.get(key) or []):
                s = str(item).strip()
                if s and s.lower() not in seen:
                    seen.add(s.lower()); out.append(s)
        return out[:limit]

    draft = []
    for theme, cands in ranked:
        canon = Counter(c["codename"].strip().upper().replace(" ", "_")
                        for c in cands).most_common(1)[0][0]
        draft.append({
            "codename": canon,
            "label": theme.title(),
            "description": max((c.get("description", "") for c in cands), key=len, default=""),
            "quantitative_signals": merge_list(cands, "signals"),
            "example_comments": merge_list(cands, "examples"),
            "_n_candidates": len(cands),   # how many raw candidates merged — your cue for importance; remove before approving
        })

    covered = sum(d["_n_candidates"] for d in draft)
    print(f"{len(raw)} candidates -> {len(groups)} themes. Drafted top {len(draft)} "
          f"(cover {covered}/{len(raw)} = {100*covered/len(raw):.0f}% of candidates):")
    for d in draft:
        print(f"  {d['codename']:<28} merged {d['_n_candidates']:>3} | {d['label']}")

    if write:
        data["final_taxonomy"] = draft
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"\n✏️  Draft written to {path} -> final_taxonomy.")
        print("    Next: MERGE semantically-similar personas, trim to a MECE set,")
        print("    remove '_n_candidates', then set status='APPROVED'.")
    return draft

# Tune top_k / min_count to taste; re-run freely (it only rewrites final_taxonomy).
_draft = draft_taxonomy_from_candidates(top_k=15)

441 candidates -> 260 themes. Drafted top 15 (cover 114/441 = 26% of candidates):
  THE_TAGGER                   merged  14 | Tagger
  THE_ADMIRER                  merged  13 | Admirer
  THE_CHEERLEADER              merged  11 | Cheerleader
  CASUAL_APPRECIATOR           merged   8 | Casual Appreciator
  SUPERFAN                     merged   7 | Superfan
  SOCIAL_CONNECTOR             merged   7 | Social Connector
  THE_APPRECIATOR              merged   7 | Appreciator
  THE_ENTHUSIAST               merged   6 | Enthusiast
  THE_CONNECTOR                merged   6 | Connector
  THE_EMPATHIZER               merged   6 | Empathizer
  ENGAGED_FAN                  merged   6 | Engaged Fan
  CASUAL_ENGAGER               merged   6 | Casual Engager
  THE_CRITIC                   merged   6 | Critic
  HUMOR_APPRECIATOR            merged   6 | Humor Appreciator
  EMOJI_REACTOR                merged   5 | Emoji Reactor

✏️  Draft written to outputs/taxonomy.json -> final_taxonomy.
    Next: MER

### Alternative — LLM Consolidation (one Gemini call, higher quality)

Instead of the string-normalised draft above, this hands ALL raw candidates to Gemini Pro and asks
for a single clean **MECE** taxonomy of ≤ `target_personas`, merging *semantic* synonyms
(`TAGGER`/`CONNECTOR`, `ADMIRER`/`APPRECIATOR`/`CHEERLEADER`) that string-matching can't. One online
call (not batch). Writes the result to `final_taxonomy`; you still review and set `status="APPROVED"`.
The two cells are alternatives — run whichever you prefer (each overwrites `final_taxonomy`).

In [19]:
from google.genai import types

def llm_consolidate_taxonomy(path=TAXONOMY_JSON_PATH, target_personas=12,
                             model=MODEL_STAGE2_CLASSIFY, write=True):
    """One-shot LLM consolidation: feed every raw candidate to Gemini and get a clean MECE taxonomy."""
    data = json.load(open(path, encoding="utf-8"))
    raw = [c for c in data.get("raw_candidates", []) if isinstance(c, dict) and c.get("codename")]
    if not raw:
        print("No raw_candidates — run/retrieve Stage 1 first.")
        return []

    items = [{"codename": c.get("codename", ""),
              "description": str(c.get("description", ""))[:220],
              "signals": (c.get("signals") or [])[:3],
              "examples": (c.get("examples") or [])[:2]} for c in raw]

    system = (
        f"You are consolidating {len(items)} NOISY candidate audience-persona archetypes — discovered "
        "batch-by-batch from an Italian Instagram influencer's comment community, with many near-duplicate "
        "names and heavy semantic overlap — into ONE clean, MECE taxonomy.\n"
        f"Merge synonyms and overlapping archetypes into AT MOST {target_personas} distinct, non-overlapping "
        "personas that together cover the candidates. Order them by how common/important the archetype is.\n"
        "For each final persona output: codename (UPPER_SNAKE_CASE), label (short Title Case), "
        "description (1-2 sentences), quantitative_signals (3-5 distinguishing behavioural signals), "
        "example_comments (2-3 verbatim fragments drawn from the candidates).\n"
        "Output ONLY a valid JSON array of objects with exactly those 5 keys. No preamble, no markdown fences."
    )
    prompt = system + "\n\n=== CANDIDATES ===\n" + json.dumps(items, ensure_ascii=False)

    cfg = types.GenerateContentConfig(
        temperature=0.0, top_p=1.0, max_output_tokens=8192,
        response_mime_type="application/json",
        thinking_config=types.ThinkingConfig(thinking_budget=4096),   # hard reasoning task — give it room
    )
    print(f"Consolidating {len(items)} candidates -> <= {target_personas} personas via {model} …")
    resp = client.models.generate_content(model=model, contents=prompt, config=cfg)

    text = strip_fences(resp.text or "")
    try:
        final = json.loads(text)
    except Exception as e:
        print("⚠️  Could not parse LLM output:", str(e)[:120], "\n", text[:400])
        return []
    if not isinstance(final, list):
        print("⚠️  Expected a JSON array, got", type(final).__name__)
        return []

    keys = ["codename", "label", "description", "quantitative_signals", "example_comments"]
    final = [{k: p.get(k, "" if k in ("codename", "label", "description") else []) for k in keys}
             for p in final if isinstance(p, dict)]

    print(f"✅ {len(final)} consolidated personas:")
    for p in final:
        print(f"  {str(p['codename']):<28} | {p['label']}")
    if write:
        data["final_taxonomy"] = final
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"\n✏️  Written to {path} -> final_taxonomy. Review/tweak, then set status='APPROVED'.")
    return final

# Uncomment to run (one online Gemini call):
_final = llm_consolidate_taxonomy(target_personas=12)

Consolidating 441 candidates -> <= 12 personas via gemini-2.5-pro …
✅ 10 consolidated personas:
  CASUAL_APPRECIATOR           | The Cheerleader
  EMOJI_REACTOR                | The Visual Reacter
  SOCIAL_CONNECTOR             | The Sharer
  SUPERFAN                     | The Loyalist
  STORYTELLER                  | The Empath
  HUMORIST                     | The Witty Observer
  INQUISITIVE_SHOPPER          | The Information Seeker
  FIRST_RESPONDER              | The Early Bird
  THOUGHTFUL_CONTRIBUTOR       | The Analyst
  CONSTRUCTIVE_CRITIC          | The Debater

✏️  Written to outputs/taxonomy.json -> final_taxonomy. Review/tweak, then set status='APPROVED'.


## Stage 2 — Deterministic Classification (Batch · Multimodal)

**One user per JSONL line.** Each request carries the approved taxonomy (system text), the user's
behavioural profile, and their engaged-post media. Pro returns exactly one persona per user with a
confidence and a cited justification; we join back on the `author_id` the model echoes.

In [10]:
def build_stage2_system_prompt(taxonomy: list) -> str:
    taxonomy_text = ""
    for p in taxonomy:
        taxonomy_text += (
            f"\nPERSONA: {p['codename']} — {p.get('label', '')}\n"
            f"  Description: {p.get('description', '')}\n"
            f"  Quantitative signals: {'; '.join(p.get('quantitative_signals', []))}\n"
            f"  Example comments: {' | '.join(p.get('example_comments', []))}\n"
        )
    return (
        "You are a deterministic community analyst for Show Reel Media Group.\n"
        "Classify the Instagram commenter into exactly ONE persona from the approved taxonomy,\n"
        "using their behavioural metrics, comment samples, AND the attached post media (images /\n"
        "video frames + transcript) of the content they engage with most.\n\n"
        "=== APPROVED PERSONA TAXONOMY ===\n"
        f"{taxonomy_text}"
        "=================================\n\n"
        "RULES:\n"
        "1. Assign exactly ONE persona — the closest match.\n"
        "2. Output a confidence score between 0.0 and 1.0.\n"
        "3. Cite a specific comment fragment or media detail as justification.\n"
        "4. If data is insufficient, assign the most probable persona with confidence <= 0.4.\n"
        "5. Echo the author_id exactly as given.\n"
        "6. Output ONLY a single valid JSON object. No preamble, no markdown fences.\n\n"
        'Schema: {"author_id": str, "persona_codename": str, "confidence": float, "justification": str}'
    )

def format_user_profile_for_stage2(row) -> dict:
    return {
        "author_id":                str(row["author_id"]),
        "total_comments":           int(row["total_comments"]),
        "unique_posts":             int(row["unique_posts_commented"]),
        "activity_span_days":       int(row["activity_span_days"]),
        "pct_comments_under_1h":    round(float(row["pct_comments_under_1h"]), 2),
        "pct_comments_under_24h":   round(float(row["pct_comments_under_24h"]), 2),
        "reply_ratio":              round(float(row["reply_ratio"]), 2),
        "mean_mention_count":       round(float(row["mean_mention_count"]), 2),
        "mean_word_count":          round(float(row["mean_word_count"]), 1),
        "emoji_usage_rate":         round(float(row["emoji_usage_rate"]), 2),
        "question_rate":            round(float(row["question_rate"]), 2),
        "exclamation_rate":         round(float(row["exclamation_rate"]), 2),
        "post_concentration_ratio": round(float(row["post_concentration_ratio"]), 2),
        "sample_comments":          str(row.get("top_comments_sample", ""))[:500],
    }

def build_stage2_line(row, system_prompt: str) -> dict:
    profile = json.dumps(format_user_profile_for_stage2(row), ensure_ascii=False)
    parts = [{"text": system_prompt + "\n\n=== USER TO CLASSIFY ===\n" + profile +
              "\n\nThe images/frames and transcripts below are sample posts this user engaged with most."}]
    parts.extend(build_user_media_parts(row["author_id"]))
    parts.append({"text": "\nClassify THIS user into exactly one persona. Output ONLY one JSON object."})
    return {"request": {"contents": [{"role": "user", "parts": parts}],
                        "generationConfig": gen_config_dict(MAX_OUTPUT_TOKENS_STAGE2, THINKING_BUDGET_STAGE2)}}

# ── SUBMIT (returns immediately — safe to close the laptop) ────────────────────
def submit_stage2(user_features_df, taxonomy, max_users=STAGE2_MAX_USERS):
    df = user_features_df if max_users is None else user_features_df.head(max_users)
    print(f"\n{'='*60}\nSTAGE 2 (batch submit) — Classification\n"
          f"  Users: {len(df):,} (1/request) | Model: {MODEL_STAGE2_CLASSIFY}\n{'='*60}")
    system_prompt = build_stage2_system_prompt(taxonomy)
    lines = [build_stage2_line(row, system_prompt)
             for _, row in tqdm(df.iterrows(), total=len(df), desc="Build Stage 2 requests")]
    in_uri  = upload_to_gcs(write_jsonl(lines, f"{LOCAL_DIR}/stage2_input.jsonl"),
                            GCS_BUCKET, BATCH_INPUT_PREFIX + "stage2_input.jsonl")
    out_uri = f"gs://{GCS_BUCKET}/{BATCH_OUTPUT_PREFIX}stage2/"
    job = submit_batch_job(in_uri, out_uri, MODEL_STAGE2_CLASSIFY)
    record_batch_job("stage2", job, out_uri)
    print(f"\n\U0001f4e4 Stage 2 submitted ({len(lines)} requests). Safe to close the laptop.")
    print("   When it finishes, run:  retrieve_stage2()   -> writes user_personas.parquet")
    return job

# ── RETRIEVE (run later; rebuilds the per-user join from user_features) ────────
def retrieve_stage2(user_features_df=None, max_users=STAGE2_MAX_USERS, output_path=RESULTS_PATH):
    job = get_recorded_job("stage2")
    if job.state != JobState.JOB_STATE_SUCCEEDED:
        print(f"⏳ Stage 2 not ready (state={job.state}). Re-run later.")
        return None
    if user_features_df is None:
        user_features_df = user_features
    df = user_features_df if max_users is None else user_features_df.head(max_users)

    results = []
    for t in retrieve_response_texts(job, GCS_BUCKET):
        if not t:
            continue
        try:
            obj = json.loads(strip_fences(t))
            if isinstance(obj, list):
                results.extend(obj)
            elif isinstance(obj, dict):
                results.append(obj)
        except Exception as e:
            print("   parse error:", str(e)[:80])

    results_df = pd.DataFrame(results)
    if "author_id" in results_df.columns:
        results_df["author_id"] = results_df["author_id"].astype(str)
    summary_cols = ["author_id", "total_comments", "activity_span_days",
                    "mean_hours_to_comment", "pct_comments_under_1h",
                    "reply_ratio", "mean_word_count"]
    base = df.copy()
    base["author_id"] = base["author_id"].astype(str)
    final_df = base[summary_cols].merge(results_df, on="author_id", how="left")
    final_df.to_parquet(output_path, index=False)

    matched = final_df["persona_codename"].notna().sum() if "persona_codename" in final_df.columns else 0
    print(f"\n✅ Stage 2 complete. Classified {matched:,}/{len(final_df):,} users -> {output_path}")
    if "persona_codename" in final_df.columns:
        dist = final_df["persona_codename"].value_counts(normalize=True).mul(100).round(1)
        print("\n   Persona Distribution:")
        for persona, pct in dist.items():
            print(f"      {str(persona):<35} {pct:.1f}%")
    return final_df

print("✅ Stage 2 functions ready (submit_stage2 / retrieve_stage2).")

✅ Stage 2 functions ready (submit_stage2 / retrieve_stage2).


## Preview a Request (no submit)

Build the **first** request of each stage exactly as it will be sent — and print its parts (profile text + attached `gs://` media) — **without** uploading or submitting anything. Use this to sanity-check the multimodal payload (media count, transcript, metadata) before spending a batch job.

In [ ]:
# === Preview the first request of each stage, WITHOUT submitting ===
def _print_parts(parts):
    print(f"  {len(parts)} part(s):")
    n_media = 0
    for p in parts:
        if "text" in p:
            t = p["text"].strip().replace(chr(10), " ⏎ ")
            print("   text :", (t[:160] + ("…" if len(t) > 160 else "")) if t else "(empty)")
        elif "fileData" in p:
            n_media += 1
            print("   media:", p["fileData"]["fileUri"])
    print(f"  → {n_media} media part(s) attached.")

# ── Stage 1 preview (grouped users + their engaged-post media) ──────────────
print("="*64)
print("STAGE 1 preview — Taxonomy Discovery")
print("="*64)
_s1 = user_features.sample(n=min(SAMPLE_N_USERS, len(user_features)),
                           random_state=SAMPLE_SEED).reset_index(drop=True)
_g0 = _s1.iloc[:STAGE1_USERS_PER_REQUEST]
_line1 = build_stage1_line(_g0)
print(f"request[0] groups {len(_g0)} user(s) | model={MODEL_STAGE1_EXPLORATORY} | "
      f"genCfg={_line1['request']['generationConfig']}")
_print_parts(_line1["request"]["contents"][0]["parts"])
print("  authors in this request:", list(_g0["author_id"].astype(str)))

# ── Stage 2 preview (1 user/request) — needs an APPROVED taxonomy ───────────
print("\n" + "="*64)
print("STAGE 2 preview — Classification")
print("="*64)
try:
    _tax = load_approved_taxonomy(TAXONOMY_JSON_PATH)
    _row = user_features.iloc[0]
    _sys = build_stage2_system_prompt(_tax)
    _line2 = build_stage2_line(_row, _sys)
    print(f"request[0] = author {_row['author_id']} | model={MODEL_STAGE2_CLASSIFY} | "
          f"genCfg={_line2['request']['generationConfig']}")
    _print_parts(_line2["request"]["contents"][0]["parts"])
except FileNotFoundError:
    print("⏭️  No taxonomy.json yet — run Stage 1 + approve before previewing Stage 2.")
except ValueError as _e:
    print(f"⏭️  Stage 2 preview skipped: {_e}")


## Pipeline Orchestrator

Single entry point for the Colab Enterprise scheduler. `run_pipeline` submits real batch jobs and
blocks on polling until each completes.

In [ ]:
def load_approved_taxonomy(path=TAXONOMY_JSON_PATH) -> list:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if data.get("status") != "APPROVED":
        raise ValueError(f"Taxonomy at '{path}' is not approved. Review and set status='APPROVED'.")
    taxonomy = data.get("final_taxonomy", [])
    if not taxonomy:
        raise ValueError("final_taxonomy is empty.")
    print(f"✅ Approved taxonomy loaded: {len(taxonomy)} personas.")
    return taxonomy

def run_pipeline(mode=PIPELINE_MODE):
    """Submit-only driver. Each stage fires its batch job and returns immediately — then you
    close the laptop and run retrieve_stage1() / retrieve_stage2() later."""
    print(f"\n{'#'*60}\n  SHOW REEL PERSONA PIPELINE (BATCH submit) — MODE: {mode}\n{'#'*60}")
    if mode in ("SAMPLE", "ALL"):
        submit_stage1(user_features)
        if mode == "SAMPLE":
            return None
    if mode in ("FULL", "ALL"):
        taxonomy = load_approved_taxonomy(TAXONOMY_JSON_PATH)
        submit_stage2(user_features, taxonomy)
    return None

result = run_pipeline(PIPELINE_MODE)


############################################################
  SHOW REEL PERSONA PIPELINE (BATCH submit) — MODE: SAMPLE
############################################################

STAGE 1 (batch submit) — Taxonomy Discovery
  Sample: 500 users | 5/request | Model: gemini-2.5-flash


Build Stage 1 requests: 100%|██████████| 100/100 [08:21<00:00,  5.01s/it]


[prep] wrote 100 requests -> outputs/stage1_input.jsonl
[upload] outputs/stage1_input.jsonl -> gs://afb_showreel/persona_batch/input/stage1_input.jsonl
[submit] gemini-2.5-flash -> projects/840606707685/locations/us-central1/batchPredictionJobs/5683232100526850048  (JOB_STATE_PENDING)
[record] stage1 job -> outputs/stage1_job.json

📤 Stage 1 submitted (100 requests). Safe to close the laptop.
   When it finishes, run:  retrieve_stage1()   -> writes taxonomy.json for review


## Retrieve a Finished Batch (run anytime — even after closing & reopening)

Submit is non-blocking, so the Vertex job keeps running on Google's side and writes to
`gs://afb_showreel/persona_batch/output/...` even with your laptop off. Come back later and run the
matching retrieve — it re-fetches the job by the id saved in `outputs/<stage>_job.json`, and only
does work once the job is `SUCCEEDED`:
- **`retrieve_stage1()`** → writes `outputs/taxonomy.json` for review
- **`retrieve_stage2()`** → writes `outputs/user_personas.parquet`

In [15]:
# Status check (no retrieval) — see whether each submitted job is done yet.
for _tag in ("stage1", "stage2"):
    if os.path.exists(f"{LOCAL_DIR}/{_tag}_job.json"):
        try:
            get_recorded_job(_tag)
        except Exception as _e:
            print(_tag, "->", str(_e)[:120])
    else:
        print(f"{_tag}: not submitted yet.")

# When the job is SUCCEEDED, run the matching line:
retrieve_stage1()
# retrieve_stage2()

[stage1] projects/840606707685/locations/us-central1/batchPredictionJobs/5683232100526850048 -> JOB_STATE_SUCCEEDED
stage2: not submitted yet.
[stage1] projects/840606707685/locations/us-central1/batchPredictionJobs/5683232100526850048 -> JOB_STATE_SUCCEEDED
[retrieve] 200 response rows from 2 shard(s)
   parse error: Unterminated string starting at: line 6 column 7 (char 248)
   parse error: Expecting value: line 6 column 35 (char 277)
   parse error: Unterminated string starting at: line 7 column 7 (char 249)
   parse error: Unterminated string starting at: line 6 column 7 (char 281)
   parse error: Unterminated string starting at: line 7 column 7 (char 300)
   parse error: Unterminated string starting at: line 7 column 7 (char 260)
   parse error: Expecting value: line 7 column 26 (char 232)
   parse error: Expecting value: line 6 column 32 (char 260)
   parse error: Unterminated string starting at: line 7 column 7 (char 262)
   parse error: Expecting value: line 6 column 33 (char 3

[{'codename': 'EMOJI_REACTOR',
  'description': 'This user primarily expresses appreciation or amusement through emojis, often with minimal or no text, and may engage with content long after its initial publication.',
  'signals': ['Avg word count: 0-2',
   'Emoji rate: 100%',
   'Reply ratio: 0%',
   'Avg hrs to comment: >300h (can be very late)'],
  'examples': ['👏👏👏😂', '😂']},
 {'codename': 'CASUAL_COMMENTER',
  'description': "This user engages occasionally with short, relevant comments, often including emojis, and sometimes directly addresses the influencer or the post's topic.",
  'signals': ['Total comments: 1-2',
   'Avg word count: 2-5',
   'Emoji rate: 50-100%',
   'Reply ratio: 0%',
   'Activity span: Long (infrequent but sustained)'],
  'examples': ['@camihawke andando dal dentista forse 🤣',
   '@viola_kurmayeva ahahaahahhaahaa']},
 {'codename': 'ENGAGED_REPLIER',
  'description': 'This user actively participates in conversations by replying to other commenters, often sharin

In [16]:
import json
from collections import Counter

# ── Stage 1: discovered candidate codenames (raw + naming-normalised) ──
tax = json.load(open(TAXONOMY_JSON_PATH, encoding="utf-8"))
raw = [c.get("codename","").strip() for c in tax.get("raw_candidates", []) if c.get("codename")]

def norm(c):                      # collapse THE/underscore/case variants
    c = c.upper().replace("_", " ").strip()
    return c[4:].strip() if c.startswith("THE ") else c

raw_cnt  = Counter(raw)
norm_cnt = Counter(norm(c) for c in raw)
print(f"raw unique codenames      : {len(raw_cnt)}  (from {len(raw)} candidates)")
print(f"normalised unique themes  : {len(norm_cnt)}")
print("\nTop themes (normalised) by share:")
tot = sum(norm_cnt.values()) or 1
for name, n in norm_cnt.most_common(20):
    print(f"  {name:<28} {n:>4}  {100*n/tot:5.1f}%")

# ── Stage 2: final persona proportions (after retrieve_stage2) ──
import os
if os.path.exists(RESULTS_PATH):
    df = pd.read_parquet(RESULTS_PATH)
    print(f"\nStage 2 — {df['persona_codename'].nunique()} unique personas over {len(df):,} users:")
    print((df['persona_codename'].value_counts(normalize=True)*100).round(1).astype(str) + ' %')

raw unique codenames      : 308  (from 441 candidates)
normalised unique themes  : 260

Top themes (normalised) by share:
  TAGGER                         14    3.2%
  ADMIRER                        13    2.9%
  CHEERLEADER                    11    2.5%
  CASUAL APPRECIATOR              8    1.8%
  SUPERFAN                        7    1.6%
  SOCIAL CONNECTOR                7    1.6%
  APPRECIATOR                     7    1.6%
  ENTHUSIAST                      6    1.4%
  CONNECTOR                       6    1.4%
  EMPATHIZER                      6    1.4%
  ENGAGED FAN                     6    1.4%
  CASUAL ENGAGER                  6    1.4%
  CRITIC                          6    1.4%
  HUMOR APPRECIATOR               6    1.4%
  EMOJI REACTOR                   5    1.1%
  CASUAL OBSERVER                 5    1.1%
  EMOTIONAL RESPONDER             5    1.1%
  LAUGHER                         5    1.1%
  RELATABLE FAN                   4    0.9%
  LOYAL FAN                       4    0.9

## Upload Outputs to GCS

Run after the pipeline completes to persist results across ephemeral runtimes.

In [ ]:
def upload_outputs_to_gcs(local_dir="outputs/"):
    from google.cloud import storage
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)
    for fname in os.listdir(local_dir):
        local_path = os.path.join(local_dir, fname)
        bucket.blob(fname).upload_from_filename(local_path)
        print(f"   Uploaded: {local_path} -> gs://{GCS_BUCKET}/{fname}")

# upload_outputs_to_gcs()

## Validation & QA

Post-classification health checks: coverage, confidence distribution, and a MECE duplicate-assignment check.

In [ ]:
def validate_classification_output(results_path=RESULTS_PATH):
    df = pd.read_parquet(results_path)
    classified   = df["persona_codename"].notna() & (df["persona_codename"] != "CLASSIFICATION_ERROR")
    coverage_pct = classified.sum() / len(df) * 100

    print("\nCLASSIFICATION QA REPORT")
    print(f"   Total users             : {len(df):,}")
    print(f"   Successfully classified : {classified.sum():,} ({coverage_pct:.1f}%)")
    print(f"   Unclassified / errors   : {(~classified).sum():,}")

    sub = df[classified].copy()
    sub["confidence"] = pd.to_numeric(sub["confidence"], errors="coerce")
    print(f"\n   Confidence  mean: {sub['confidence'].mean():.3f}  "
          f"median: {sub['confidence'].median():.3f}  "
          f"<0.40: {(sub['confidence'] < 0.4).sum():,}")

    dupes = df["author_id"].duplicated().sum()
    print(f"\n   MECE Check: {dupes} duplicate assignments ({'FAIL' if dupes > 0 else 'PASS'})")
    display(df["persona_codename"].value_counts().to_frame("count"))
    return df

# qa_df = validate_classification_output()

## Stage 3 — Macro Persona Discovery (UMAP + HDBSCAN + LLM Naming)

Takes the Stage 2 per-user micro-persona assignments and behavioral features, and discovers
**macro** audience segments through unsupervised ML. Gemini then names each segment as a
human-readable marketing persona.

**Flow:**
1. **Load & merge** — Stage 2 `user_personas.parquet` + Stage 0 `user_features` behavioral matrix.
2. **Feature matrix** — standardised numeric features + one-hot encoded micro-persona label.
3. **UMAP (×2)** — 15-D embedding for HDBSCAN; 2-D embedding for visualisation.
4. **HDBSCAN** — density-based clustering on the 15-D UMAP embedding → macro cluster labels.
5. **Cluster summaries** — dominant micro personas, mean behavioral stats, sample comments.
6. **LLM naming** — single online Gemini call → `codename`, `label`, `description`, `key_traits`, `marketing_insight` per cluster.
7. **Save** — `outputs/user_macro_personas.parquet` (per-user) + `outputs/macro_persona_names.json` (definitions).

> **Tuning knob:** `HDBSCAN_MIN_CLUSTER_SIZE` controls granularity. Re-run HDBSCAN only (no need to redo UMAP) when adjusting.

In [ ]:
!pip install -q umap-learn hdbscan scikit-learn matplotlib

In [ ]:

# ─── Stage 3 configuration ───────────────────────────────────────────────────
# UMAP — two passes: high-D for HDBSCAN, 2-D for scatter plot
UMAP_N_NEIGHBORS          = 30    # larger = more global structure
UMAP_MIN_DIST             = 0.0   # 0.0 packs clusters tighter — ideal for HDBSCAN
UMAP_N_COMPONENTS_CLUSTER = 15    # dims fed into HDBSCAN (15 is a good default)
UMAP_METRIC               = "euclidean"
UMAP_RANDOM_STATE         = 42

# HDBSCAN — tune min_cluster_size to control granularity
# Raise it → fewer, broader macro personas; lower → more granular
HDBSCAN_MIN_CLUSTER_SIZE  = 2000  # ~1% of 194k users; adjust to taste
HDBSCAN_MIN_SAMPLES       = 100
HDBSCAN_CLUSTER_SELECTION = "eom" # excess-of-mass: fewer, more stable clusters
HDBSCAN_CLUSTER_EPSILON   = 0.0   # 0 = default HDBSCAN behaviour

# LLM naming (online call — small payload, no need for batch)
MODEL_STAGE3_NAMING           = "gemini-2.5-flash"
MAX_SAMPLE_USERS_PER_CLUSTER  = 30  # comment samples per cluster fed to LLM

# Outputs
MACRO_PERSONA_PATH = f"{LOCAL_DIR}/user_macro_personas.parquet"
CLUSTER_NAMES_PATH = f"{LOCAL_DIR}/macro_persona_names.json"

print("✅ Stage 3 config loaded.")
print(f"   UMAP   : n_neighbors={UMAP_N_NEIGHBORS}  min_dist={UMAP_MIN_DIST}  "
      f"n_components_cluster={UMAP_N_COMPONENTS_CLUSTER}")
print(f"   HDBSCAN: min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}  "
      f"min_samples={HDBSCAN_MIN_SAMPLES}  method='{HDBSCAN_CLUSTER_SELECTION}'")
print(f"   LLM    : {MODEL_STAGE3_NAMING}  (online call, not batch)")


In [ ]:

import pandas as pd, numpy as np, os

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(
        f"Stage 2 results not found at '{RESULTS_PATH}'. "
        "Run retrieve_stage2() first."
    )
results_df = pd.read_parquet(RESULTS_PATH)
results_df["author_id"] = results_df["author_id"].astype(str)
print(f"Stage 2 results : {len(results_df):,} users  |  "
      f"classified: {results_df['persona_codename'].notna().sum():,}")

# Merge with full behavioral features + comment text from Stage 0 (still in memory)
extra_cols = [
    "author_id",
    "unique_posts_commented", "total_replies_made",
    "pct_comments_under_24h", "emoji_usage_rate", "question_rate",
    "exclamation_rate", "mean_mention_count", "post_concentration_ratio",
    "top_comments_sample",
]
available = [c for c in extra_cols if c in user_features.columns]
feat_ext = user_features[available].copy()
feat_ext["author_id"] = feat_ext["author_id"].astype(str)

stage3_df = results_df.merge(feat_ext, on="author_id", how="left")

# Keep only successfully classified users
stage3_df = stage3_df[
    stage3_df["persona_codename"].notna() &
    (stage3_df["persona_codename"] != "CLASSIFICATION_ERROR")
].copy().reset_index(drop=True)

print(f"Stage 3 working set: {len(stage3_df):,} users with valid micro-persona labels")
print(f"\nMicro-persona distribution:")
print(stage3_df["persona_codename"].value_counts().to_string())


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np

# Use the selected features from the feature selection cell above
NUMERIC_FEATURES = SELECTED_NUMERIC_FEATURES.copy()
print(f"Numeric features ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")

# Standardise (fill any rare NaN with column median first)
num_data = stage3_df[NUMERIC_FEATURES].fillna(stage3_df[NUMERIC_FEATURES].median())
scaler = StandardScaler()
X_num  = scaler.fit_transform(num_data)

# One-hot encode micro persona — weight ×2 so it has comparable influence to numeric block
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_ohe = ohe.fit_transform(stage3_df[["persona_codename"]])
persona_categories = ohe.categories_[0].tolist()
print(f"OHE micro-persona dims: {X_ohe.shape[1]}  →  {persona_categories}")

X = np.hstack([X_num, X_ohe * 2.0])
print(f"
✅ Feature matrix: {X.shape}  (rows=users, cols=numeric+OHE)")

In [ ]:

import umap

print(f"Running UMAP ({UMAP_N_COMPONENTS_CLUSTER}-D) for HDBSCAN input …")
print(f"  n_neighbors={UMAP_N_NEIGHBORS}  min_dist={UMAP_MIN_DIST}  "
      f"users={len(X):,}  (expect 1-3 min)")

reducer_cluster = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,           # 0.0 packs clusters tighter — better for HDBSCAN
    n_components=UMAP_N_COMPONENTS_CLUSTER,
    metric=UMAP_METRIC,
    random_state=UMAP_RANDOM_STATE,
    low_memory=True,
    verbose=True,
)
X_umap_cluster = reducer_cluster.fit_transform(X)
print(f"\n✅ UMAP clustering embedding: {X_umap_cluster.shape}")

print(f"\nRunning UMAP (2-D) for visualisation …")
reducer_viz = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=0.1,
    n_components=2,
    metric=UMAP_METRIC,
    random_state=UMAP_RANDOM_STATE,
    low_memory=True,
    verbose=False,
)
X_umap_viz = reducer_viz.fit_transform(X)
print(f"✅ UMAP 2-D embedding: {X_umap_viz.shape}")

stage3_df["umap_x"] = X_umap_viz[:, 0]
stage3_df["umap_y"] = X_umap_viz[:, 1]


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.cm as cm

personas  = stage3_df["persona_codename"].unique()
palette   = cm.get_cmap("tab20", len(personas))
color_map = {p: palette(i) for i, p in enumerate(sorted(personas))}

fig, ax = plt.subplots(figsize=(12, 9))
for p, grp in stage3_df.groupby("persona_codename"):
    ax.scatter(grp["umap_x"], grp["umap_y"],
               s=2, alpha=0.4, color=color_map[p], label=p)

ax.set_title("UMAP 2-D projection — coloured by micro persona (Stage 2 labels)", fontsize=13)
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.legend(markerscale=5, bbox_to_anchor=(1.01, 1), loc="upper left",
          fontsize=8, framealpha=0.7)
plt.tight_layout()
plt.savefig(f"{LOCAL_DIR}/umap_micro_personas.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {LOCAL_DIR}/umap_micro_personas.png")


In [ ]:

import hdbscan

print(f"Running HDBSCAN on {UMAP_N_COMPONENTS_CLUSTER}-D UMAP embedding …")
print(f"  min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}  "
      f"min_samples={HDBSCAN_MIN_SAMPLES}  "
      f"cluster_selection_method='{HDBSCAN_CLUSTER_SELECTION}'")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    cluster_selection_method=HDBSCAN_CLUSTER_SELECTION,
    cluster_selection_epsilon=HDBSCAN_CLUSTER_EPSILON,
    prediction_data=True,
)
labels = clusterer.fit_predict(X_umap_cluster)

stage3_df["macro_cluster"] = labels
unique_clusters = sorted(set(labels[labels >= 0]))
n_noise = int((labels == -1).sum())

print(f"\n✅ HDBSCAN complete.")
print(f"   Clusters found  : {len(unique_clusters)}")
print(f"   Clustered users : {(labels >= 0).sum():,}  ({100*(labels >= 0).mean():.1f}%)")
print(f"   Noise (−1)      : {n_noise:,}  ({100*n_noise/len(labels):.1f}%)")
print("\n   Cluster sizes:")
for c in unique_clusters:
    cnt = int((labels == c).sum())
    print(f"     Cluster {c:>3}: {cnt:>7,} users  ({100*cnt/len(labels):.1f}%)")

# Tip: if you get too many / too few clusters, adjust HDBSCAN_MIN_CLUSTER_SIZE in config
# and re-run this cell. No need to rerun UMAP.


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.cm as cm

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# ── Left: macro clusters ──────────────────────────────────────────────────
palette_macro = cm.get_cmap("Set1", max(len(unique_clusters), 1))
noise_color   = (0.75, 0.75, 0.75, 0.25)

noise_mask = stage3_df["macro_cluster"] == -1
if noise_mask.any():
    axes[0].scatter(stage3_df.loc[noise_mask, "umap_x"],
                    stage3_df.loc[noise_mask, "umap_y"],
                    s=1, alpha=0.15, color=noise_color, label="noise", zorder=1)

for i, c in enumerate(unique_clusters):
    mask = stage3_df["macro_cluster"] == c
    axes[0].scatter(stage3_df.loc[mask, "umap_x"],
                    stage3_df.loc[mask, "umap_y"],
                    s=2, alpha=0.5, color=palette_macro(i),
                    label=f"Cluster {c}  (n={mask.sum():,})", zorder=2)

axes[0].set_title("Macro clusters — HDBSCAN", fontsize=12)
axes[0].set_xlabel("UMAP-1"); axes[0].set_ylabel("UMAP-2")
axes[0].legend(markerscale=6, fontsize=8)

# ── Right: micro personas ─────────────────────────────────────────────────
for p, grp in stage3_df.groupby("persona_codename"):
    axes[1].scatter(grp["umap_x"], grp["umap_y"],
                    s=2, alpha=0.4, color=color_map[p], label=p)

axes[1].set_title("Micro personas — LLM (Stage 2)", fontsize=12)
axes[1].set_xlabel("UMAP-1"); axes[1].set_ylabel("UMAP-2")
axes[1].legend(markerscale=5, fontsize=7,
               bbox_to_anchor=(1.01, 1), loc="upper left")

plt.suptitle("UMAP projection: macro clusters vs. micro personas", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f"{LOCAL_DIR}/umap_macro_clusters.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {LOCAL_DIR}/umap_macro_clusters.png")


In [ ]:

def build_cluster_summary(cluster_id: int, df: pd.DataFrame) -> dict:
    sub = df[df["macro_cluster"] == cluster_id].copy()
    n   = len(sub)

    # Micro-persona composition
    persona_dist = (
        sub["persona_codename"].value_counts(normalize=True)
                               .mul(100).round(1)
                               .head(5).to_dict()
    )

    # Mean behavioral stats
    stat_cols = [c for c in [
        "total_comments", "activity_span_days", "mean_hours_to_comment",
        "pct_comments_under_1h", "reply_ratio", "mean_word_count",
        "emoji_usage_rate", "question_rate", "exclamation_rate",
    ] if c in sub.columns]
    mean_stats = sub[stat_cols].mean().round(3).to_dict()

    # Representative comment samples from top_comments_sample
    samples = []
    if "top_comments_sample" in sub.columns:
        sample_rows = sub["top_comments_sample"].dropna().sample(
            n=min(MAX_SAMPLE_USERS_PER_CLUSTER, n), random_state=42
        )
        for txt in sample_rows:
            frags = [f.strip() for f in str(txt).split("|||") if f.strip()]
            samples.extend(frags[:2])
            if len(samples) >= 30:
                break
    samples = samples[:30]

    # Stage 2 justification snippets
    justifications = []
    if "justification" in sub.columns:
        for j in sub["justification"].dropna().sample(n=min(10, n), random_state=42):
            justifications.append(str(j)[:200])

    return {
        "cluster_id":              cluster_id,
        "n_users":                 n,
        "pct_audience":            round(100 * n / max(len(df[df["macro_cluster"] >= 0]), 1), 1),
        "dominant_micro_personas": persona_dist,
        "mean_behavioral_stats":   mean_stats,
        "sample_comments":         samples,
        "sample_justifications":   justifications[:10],
    }


cluster_summaries = [build_cluster_summary(c, stage3_df) for c in unique_clusters]
print(f"✅ Cluster summaries built for {len(cluster_summaries)} clusters:\n")
for s in cluster_summaries:
    top_p = list(s["dominant_micro_personas"].items())[:3]
    print(f"  Cluster {s['cluster_id']:>3}  n={s['n_users']:>7,}  ({s['pct_audience']:.1f}%)"
          f"  top micro-personas: {top_p}")


In [ ]:

from google.genai import types as genai_types

STAGE3_NAMING_SYSTEM = (
    "You are a strategic audience analyst for an Italian influencer marketing agency.\n"
    "You have clustered Instagram commenters into macro audience segments using UMAP + HDBSCAN.\n"
    "Each cluster is described by its dominant micro-persona mix, mean behavioral stats, "
    "and representative comment samples.\n\n"
    "For EACH cluster produce ONE macro persona with exactly these fields:\n"
    "  cluster_id        : (integer — echo back exactly as provided)\n"
    "  codename          : UPPER_SNAKE_CASE concise name (e.g. PASSIONATE_LOYALISTS)\n"
    "  label             : Short Title Case marketing label (3-5 words)\n"
    "  description       : 2-3 sentences — who they are, how and why they engage\n"
    "  key_traits        : JSON list of 4-6 behavioural / attitudinal bullet strings\n"
    "  marketing_insight : 1 actionable sentence for the brand or agency\n\n"
    "Output ONLY a valid JSON array of objects with exactly those 6 keys. "
    "No preamble, no markdown fences, no extra keys."
)

def build_naming_prompt(summaries: list) -> str:
    blocks = []
    for s in summaries:
        stats_str = "  ".join(
            f"{k}={v:.2f}" if isinstance(v, float) else f"{k}={v}"
            for k, v in list(s["mean_behavioral_stats"].items())[:7]
        )
        persona_str = "  ".join(f"{p}: {pct}%" for p, pct in s["dominant_micro_personas"].items())
        comments_str = "\n    • ".join(s["sample_comments"][:8])
        justs = s.get("sample_justifications", [])
        just_str = ("\n  LLM justification samples: " +
                    " | ".join(justs[:3])) if justs else ""
        blocks.append(
            f"CLUSTER {s['cluster_id']}  (n={s['n_users']:,}, {s['pct_audience']}% of clustered audience)\n"
            f"  Dominant micro-personas: {persona_str}\n"
            f"  Mean behavioral stats:   {stats_str}{just_str}\n"
            f"  Sample comments:\n    • {comments_str}"
        )
    return STAGE3_NAMING_SYSTEM + "\n\n=== CLUSTER DATA ===\n\n" + "\n\n".join(blocks)

prompt = build_naming_prompt(cluster_summaries)

cfg = genai_types.GenerateContentConfig(
    temperature=0.0,
    top_p=1.0,
    max_output_tokens=4096,
    response_mime_type="application/json",
    thinking_config=genai_types.ThinkingConfig(thinking_budget=2048),
)

print(f"Calling {MODEL_STAGE3_NAMING} to name {len(cluster_summaries)} macro clusters …")
resp = client.models.generate_content(model=MODEL_STAGE3_NAMING, contents=prompt, config=cfg)

raw = strip_fences(resp.text or "")
try:
    macro_personas = json.loads(raw)
except json.JSONDecodeError as e:
    print("⚠️  JSON parse error:", e)
    print("Raw response (first 800 chars):\n", raw[:800])
    macro_personas = []

if not isinstance(macro_personas, list):
    print("⚠️  Expected JSON array, got:", type(macro_personas).__name__)
    macro_personas = []

print(f"\n✅ {len(macro_personas)} macro personas named:\n")
for mp in macro_personas:
    print(f"  Cluster {mp.get('cluster_id', '?'):>3}  "
          f"{str(mp.get('codename', '')):.<40}  {mp.get('label', '')}")
    print(f"           {mp.get('description', '')[:130]}")
    print(f"           Insight: {mp.get('marketing_insight', '')[:130]}\n")

with open(CLUSTER_NAMES_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "status":        "COMPLETE",
        "n_clusters":    len(unique_clusters),
        "noise_users":   int(n_noise),
        "macro_personas": macro_personas,
    }, f, ensure_ascii=False, indent=2)
print(f"✅ Macro persona definitions saved -> {CLUSTER_NAMES_PATH}")


In [ ]:

# Build cluster_id -> macro persona lookup
id_to_macro = {}
for mp in macro_personas:
    cid = mp.get("cluster_id")
    if cid is not None:
        id_to_macro[int(cid)] = {
            "macro_persona_codename": mp.get("codename", f"CLUSTER_{cid}"),
            "macro_persona_label":    mp.get("label", ""),
        }

stage3_df["macro_persona_codename"] = stage3_df["macro_cluster"].map(
    lambda c: id_to_macro.get(c, {}).get("macro_persona_codename", "NOISE")
)
stage3_df["macro_persona_label"] = stage3_df["macro_cluster"].map(
    lambda c: id_to_macro.get(c, {}).get("macro_persona_label", "Noise / Unclustered")
)

# Save per-user macro assignments
output_cols = [
    "author_id", "persona_codename", "confidence",
    "macro_cluster", "macro_persona_codename", "macro_persona_label",
    "umap_x", "umap_y",
]
output_cols = [c for c in output_cols if c in stage3_df.columns]
stage3_df[output_cols].to_parquet(MACRO_PERSONA_PATH, index=False)

print(f"✅ Macro persona assignments saved -> {MACRO_PERSONA_PATH}")
print(f"\nMacro Persona Distribution (clustered users only):")
clustered = stage3_df[stage3_df["macro_cluster"] >= 0]
dist = (clustered["macro_persona_codename"].value_counts(normalize=True).mul(100).round(1))
for codename, pct in dist.items():
    cid = int(clustered.loc[clustered["macro_persona_codename"] == codename, "macro_cluster"].iloc[0])
    label = id_to_macro.get(cid, {}).get("macro_persona_label", "")
    print(f"  {str(codename):<42} {pct:>6.1f}%  —  {label}")
print(f"\n  Noise / unclustered: {(stage3_df['macro_cluster'] == -1).sum():,} users")
